In [1]:
%pip install "numpy" "opencv-python" -q
%pip install git+https://www.github.com/mouseland/cellpose.git
%pip install pyocclient -q

  Cloning https://www.github.com/mouseland/cellpose.git to /tmp/pip-req-build-wdx8el09
  Running command git clone --filter=blob:none --quiet https://www.github.com/mouseland/cellpose.git /tmp/pip-req-build-wdx8el09
  Resolved https://www.github.com/mouseland/cellpose.git to commit a9f8bfcde43033247309e3982747df9fe9f09315
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 90.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 105.0 MB/s eta 0:00:0000:0100:01
  Created wheel for cellpose: filename=cellpose-4.1.1-py3-none-any.whl size=213288 sha256=fa81d35aca7f0717c259088da2aaeb15c6c3d4b1c6b5c87db3eb1d8cbd33dc4a
  Stored in directory: /tmp/pip-ephem-wheel-cache-wl0udbho/wheels/df/b6/31/a3013c44290eabb46f4c06d1efb19744124fcad2d59684ec5e
Successfully built cellpose
  Preparing metadata (setup.py) ... done


In [2]:
import owncloud, getpass, cellpose

# Config
url, user = 'https://cloud.minesparis.psl.eu', 'gabriel.gautier'
oc_session = owncloud.Client(url)
oc_session.login(user, getpass.getpass(f"PW {user}: "))
oc_session.get_file('/travail/Mines/DIMA/Segmentation/scripts/tool.py', 'tool.py')

import tool
oc = tool.Owncloud(oc_session)

print("Tool chargé et prêt.")

Tool chargé et prêt.


In [3]:
oc.upload_data()

In [4]:
oc.upload_scripts()

In [5]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tool


from cellpose import core, utils, io, models, metrics, train, dynamics, transforms, plot
from glob import glob

import importlib
importlib.reload(tool)
import tool

visualizer = tool.CellVisualizer()

In [6]:

oc.upload_models()

In [7]:
ids_test = [str(i).zfill(3) for i in range(1,326)]

In [10]:

from cellpose import models, io

bench_configs = [
    {
        "label": "raw_2channel_cellpose4",
        "target": "Cell",
        "load_method": "channel",
        "mode": "raw",
        "params": {"channels": [1, 2], "flow_threshold": 0.5, "cellprob_threshold": 0.0},
        "model_path": "./models/cyto4_40_raw.pth" 
    },
    {
        "label": "Nuclei_cellpose4",
        "target": "Nuc",
        "load_method": "data",
        "mode": "Image",
        "params": {"channels": [0, 0], "flow_threshold": 0.4, "cellprob_threshold": -0.5},
        "model_path": "./models/cellpose4_nuclei.pth"
    }
]

# Création du dossier racine 
base_output = os.path.abspath("./results/SB_test")
os.makedirs(base_output, exist_ok=True)

for config in bench_configs:
    print(f"\nInférence : {config['label']}...")
    
    model_root = os.path.join(base_output, config['label'])
    output_dir_masks = os.path.join(model_root, "masks")
    output_dir_viz = os.path.join(model_root, "viz")
    
    os.makedirs(output_dir_masks, exist_ok=True)
    os.makedirs(output_dir_viz, exist_ok=True)
    
    model = models.CellposeModel(gpu=True, pretrained_model=config['model_path'])
    
    if config['load_method'] == "channel":
        X_val_multi, _ = visualizer.load_channel(ids_test, mode=config['mode'])

    for idx, img_id in enumerate(ids_test):
        # Chargement de l'image uniquement
        if config['load_method'] == "channel":
            img = X_val_multi[idx]
        else:
            img = visualizer.load_data(img_id, mode=config['mode'])
            
        # Inférence Cellpose
        masks, _, _ = model.eval(img, **config['params'])
        
        # Sauvegarde du masque
        save_path_mask = os.path.join(output_dir_masks, f"{img_id}_{config['target']}_pred.bmp")
        cv2.imwrite(save_path_mask, masks.astype(np.uint16)) 
        
        # Grille de visualisation individuelle (1 ligne, 2 colonnes)
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        ax1, ax2 = axes.flatten()
        
        # Affichage Image Entrée
        visualizer.plot(i=img_id, mode='Image', ax=ax1, show=False)
        ax1.set_title("A. Image Entrée")

        visualizer.plot_points_overlay(i=img_id, mask_pred=masks, ax=ax2, show=False, annotation=False, prediction=True)
        ax2.set_title(f"B. Prédiction Cellpose - {config['target']})")
        
        titre_rapport = f"Résultat Inférence : {img_id} ({config['label']})"
        fig.suptitle(titre_rapport, fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        fig.subplots_adjust(top=0.90)
        
        save_path_viz = os.path.join(output_dir_viz, f"{img_id}_viz.png")
        plt.savefig(save_path_viz, dpi=150, bbox_inches='tight')
        plt.close(fig)

# --- EXPORT GLOBAL ---
print("\nExportation du dossier d'inférence...")
try:
    chemin_export_global = "/travail/Mines/DIMA/Segmentation/data/prediction/benchmark_test_SB"
    oc.download_path(base_output, chemin_export_global)
    print(f"Export réussi vers : {chemin_export_global}")
except Exception as e:
    print(f"Erreur lors de l'export global : {e}")

print(f"\nProcessus terminé. Les résultats sont sauvegardés dans : {base_output}")


Inférence : raw_2channel_cellpose4...



Inférence : Nuclei_cellpose4...



Exportation du dossier d'inférence...
Erreur lors de l'export global : HTTP error: 504

Processus terminé. Les résultats sont sauvegardés dans : /content/results/SB_test
